<a href="https://colab.research.google.com/github/budennovsk/Pandas/blob/master/v2_cold_start.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install implicit catboost rectools[lightfm] replay-rec==0.21.2rc0 faiss-cpu #replay-rec !pip install faiss-cpu
# !pip -q uninstall -y pyspark
# !pip -q install "pyspark==3.4.3"


In [ ]:
from pathlib import Path
import typing as tp
import warnings

import pandas as pd
import numpy as np

from implicit.nearest_neighbours import CosineRecommender
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import RidgeClassifier
from catboost import CatBoostClassifier, CatBoostRanker
try:
    from lightgbm import LGBMClassifier, LGBMRanker
    LGBM_AVAILABLE = True
except ImportError:
    warnings.warn("lightgbm is not installed. Some parts of the notebook will be skipped.")
    LGBM_AVAILABLE = False
from rectools.dataset import Interactions
from lightfm import LightFM
from rectools import Columns
from rectools.dataset import Dataset
from rectools.metrics import Precision, Recall, MeanInvUserFreq, Serendipity,MAP,NDCG,HitRate,calc_metrics
from rectools.models import (
    ImplicitALSWrapperModel,
    ImplicitBPRWrapperModel,
    LightFMWrapperModel,
    PureSVDModel,
    ImplicitItemKNNWrapperModel,
    EASEModel,
    RandomModel,
    PopularInCategoryModel,
    PopularModel)
from implicit.als import AlternatingLeastSquares
from implicit.bpr import BayesianPersonalizedRanking
from implicit.nearest_neighbours import CosineRecommender
from rectools.models.base import ExternalIds
from rectools.models.ranking import (
    CandidateRankingModel,
    CandidateGenerator,
    Reranker,

    CatBoostReranker,
    CandidateFeatureCollector,
    PerUserNegativeSampler
)
from rectools.model_selection import cross_validate, TimeRangeSplitter,LastNSplitter,Splitter
from tqdm.auto import tqdm

In [ ]:
path_users = '/content/drive/MyDrive/Colab Notebooks/Симбирсофт/recsys/dataset/data_original/users.csv'
path_items = '/content/drive/MyDrive/Colab Notebooks/Симбирсофт/recsys/dataset/data_original/items.csv'
path_interactions = '/content/drive/MyDrive/Colab Notebooks/Симбирсофт/recsys/dataset/data_original/interactions.csv'


users = pd.read_csv(path_users)
items = pd.read_csv(path_items)
interactions = (
    pd.read_csv(path_interactions, parse_dates=["last_watch_dt"])
    .rename(columns={"last_watch_dt": Columns.Datetime})
)
interactions = interactions.head(1000000)
users_clise = users[users['user_id'].isin(interactions['user_id'].unique())]
items_clise = items[items['item_id'].isin(interactions['item_id'].unique())]
interactions["weight"] = 1
dataset = Dataset.construct(interactions)
RANDOM_STATE = 32


# dataset
# interactions[Columns.Datetime] = pd.to_datetime(interactions[Columns.Datetime], format='%Y-%m-%d')

In [ ]:
max_date = interactions[Columns.Datetime].max()
train = interactions[interactions[Columns.Datetime] < max_date - pd.Timedelta(days=7)].copy()
test = interactions[interactions[Columns.Datetime] >= max_date - pd.Timedelta(days=7)].copy()

print(f"train: {train.shape}")
print(f"test: {test.shape}")

In [ ]:
# отфильтруем холодных пользователей из теста
cold_users = set(test[Columns.User]) - set(train[Columns.User])
len(cold_users)

In [ ]:
test['datetime'].min(), test['datetime'].max()

In [ ]:
train[train['user_id'].isin(cold_users)]['user_id'].nunique()

In [ ]:
test_cold=test[test['user_id'].isin(cold_users)]
test_cold[test_cold['user_id'].isin(cold_users)]['user_id'].nunique()

In [ ]:
valid_user_ids = set(users_clise["user_id"].dropna().unique())
valid_item_ids = set(items_clise["item_id"].dropna().unique())

train_in_all = train[
    train["user_id"].isin(valid_user_ids) &
    train["item_id"].isin(valid_item_ids)
].copy()

train_in_all['user_id']
users_clise_train = users_clise[users_clise['user_id'].isin(train_in_all['user_id'].unique())]
# users_clise_train['user_id'].nunique()

items_clise_train = items_clise[items_clise['item_id'].isin(train_in_all['item_id'].unique())]
# items_clise_train['item_id'].nunique()

In [ ]:
# train_dataset_all = Dataset.construct(interactions_df=train_in_all)

In [ ]:
items_clise_train["genre"] = items_clise_train["genres"].str.split(",")
genre_feature = items_clise_train[["item_id", "genre"]].explode("genre")
genre_feature.columns = ["id", "value"]
genre_feature["feature"] = "genre"
genre_feature_train = genre_feature[genre_feature['id'].isin(train['item_id'])]
genre_feature_train.head()

In [ ]:
import re

col = "value"

def canon(x: str) -> str:
    x = str(x).strip().lower()
    x = re.sub(r"\s+", " ", x)
    x = x.replace("–", "-").replace("—", "-")
    x = re.sub(r"\s*-\s*", "-", x)  # "ток - шоу" -> "ток-шоу"
    return x

# 1) нормализуем текст
genre_feature_train[col] = genre_feature_train[col].map(canon)

# 2) склейки (синонимы/варианты)
# mapping = {
#     "советские": "русские",
#     # (по написанию)
#     "токшоу": "ток-шоу",
#     "реалити": "реалити-шоу",
#     "шоу": "телешоу",  # если хочешь объединять "шоу" с "телешоу"
#     "мультфильм": "мультфильмы",
#     "русские мультфильмы": "мультфильмы",  # если хочешь все мультфильмы в одну категорию
#     "западные мультфильмы": "мультфильмы",
#     "детские": "для детей",
#     "для самых маленьких": "для детей",    # если хочешь объединять
# }
mapping = {
    "советские": "русские",
    # (по написанию)
    "токшоу": "ток-шоу",
    "реалити": "реалити-шоу",
    "шоу": "телешоу",  # если хочешь объединять "шоу" с "телешоу"
    "мультфильм": "мультфильмы",
    "русские мультфильмы": "мультфильмы",  # если хочешь все мультфильмы в одну категорию
    "западные мультфильмы": "мультфильмы",
    "детские": "для детей",
    "для самых маленьких": "для детей",
    "исторические":	"историческое",
    "музыка":	"музыкальные",
    "музыка":	"мюзиклы",
    "комиксы":"по комиксам",
    "спорт":"футбол",
    "комедии":	"юмор",
}

genre_feature_train[col] = genre_feature_train[col].replace(mapping)
genre_feature_train[col].nunique()

In [ ]:

counts = (
    genre_feature_train
    .groupby('value', dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
)

counts_lt10 = counts[counts["count"] < 10].reset_index(drop=True)

counts_lt10  # все value, которые встречаются меньше 10 раз

In [ ]:
rare_values = set(counts_lt10["value"])

genre_feature_train["value"] = genre_feature_train["value"].where(
    ~genre_feature_train["value"].isin(rare_values),
    "other",
)
genre_feature_train[col].nunique()

In [ ]:
# import re
# import numpy as np
# import pandas as pd
# from sentence_transformers import SentenceTransformer
# from sklearn.metrics.pairwise import cosine_similarity

# df = genre_feature_train
# col = "value"

# def canon(x: str) -> str:
#     x = str(x).strip().lower()
#     x = re.sub(r"\s+", " ", x)
#     x = x.replace("–", "-").replace("—", "-")
#     x = re.sub(r"\s*-\s*", "-", x)
#     return x

# # уникальные жанры
# genres = sorted(pd.Series(df[col]).map(canon).dropna().unique().tolist())

# model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
# emb = model.encode(genres, normalize_embeddings=True)

# sim = cosine_similarity(emb)

# threshold = 0.72  # подбирается: 0.68–0.80 (выше = строже)
# pairs = []
# n = len(genres)
# for i in range(n):
#     for j in range(i+1, n):
#         s = float(sim[i, j])
#         if s >= threshold:
#             pairs.append((genres[i], genres[j], s))

# pairs = sorted(pairs, key=lambda x: -x[2])
# pairs_df = pd.DataFrame(pairs, columns=["a", "b", "sim"])

# # посмотри топ совпадений
# pairs_df.head(50)

In [ ]:
# dataset_feature_train_all = Dataset.construct(
#     interactions_df=train_in_all,
#     user_features_df=None,
#     item_features_df=genre_feature_train,
#     cat_item_features=['genre']
# )

In [ ]:
def plot_normalized_barh_chart(
    dataframe: pd.DataFrame,
    column_name: str,
    title: str,
    figsize: tp.Tuple[int, int] = (12, 7),
    annotate_chart: bool = True,
    xlabel: tp.Optional[str] = None,
    ylabel: tp.Optional[str] = None,
):
    ax = (
        dataframe[column_name]
        .value_counts(dropna=False, normalize=True)
        .sort_index()
        .plot(
            kind='barh',
            grid=True,
            title=title,
            figsize=figsize,
            xlabel=xlabel,
            ylabel=ylabel,
        )
    )

    if annotate_chart:
        for bars in ax.containers:
            ax.bar_label(bars, labels=[f'{x:.1%}' for x in bars.datavalues])

In [ ]:
users_clise_train["sex"] = users_clise_train["sex"].map({"Ж": 1, "М": 0}).fillna(-1).astype("int8")
age_category = pd.CategoricalDtype(
    categories=[
         "unknown",
        'age_18_24',
        'age_25_34',
        'age_35_44',
        'age_45_54',
        'age_55_64',
        'age_65_inf',
    ],
    ordered=True,
)
users_clise_train["age"] = (
    users_clise_train["age"]
    .fillna("unknown")
    .astype(age_category)
)
income_category = pd.CategoricalDtype(
    categories=[
        "unknown",          # добавили
        "income_0_20",
        "income_20_40",
        "income_40_60",
        "income_60_90",
        "income_90_150",
        "income_150_inf",
    ],
    ordered=True,
)

# если income уже строки/категории — заполним nan и приведем тип
users_clise_train["income"] = (
    users_clise_train["income"]
    .fillna("unknown")
    .astype(income_category)
)
YEAR_FROM = 1990
STEP_SIZE = 5
bins = [year for year in range(YEAR_FROM, int(items_clise['release_year'].max()) + STEP_SIZE, STEP_SIZE)]
bins = [int(items_clise['release_year'].min())] + bins
items_clise['year_bin'] = pd.cut(items_clise['release_year'],
                           bins=bins, include_lowest=True)

In [ ]:
users_clise_train['age'].value_counts()

In [ ]:
plot_normalized_barh_chart(users_clise_train, 'sex', 'Распределение выхода контента')

In [ ]:
user_features_frames = []
for feature in ["sex", "age", "income"]:
    feature_frame = users_clise_train.reindex(columns=[Columns.User, feature])
    feature_frame.columns = ["id", "value"]
    feature_frame["feature"] = feature
    user_features_frames.append(feature_frame)
user_features_train = pd.concat(user_features_frames)
user_features_train.tail()

In [ ]:
content_feature_train = items_clise_train.reindex(columns=[Columns.Item, "content_type"])
content_feature_train.columns = ["id", "value"]
content_feature_train["feature"] = "content_type"

In [ ]:
item_features_train = pd.concat((genre_feature_train,content_feature_train))
item_features_train

In [ ]:
item_features_train,user_features_train

In [ ]:
dataset_train_i_u_features = Dataset.construct(
    interactions_df=train_in_all,
    user_features_df=user_features_train,
    cat_user_features=["sex", "age", "income"],
    item_features_df=item_features_train,
    cat_item_features=["genre", "content_type"],
)

In [ ]:
# Take few models to compare
from datetime import timedelta

models = {
    "popular": PopularModel(popularity="n_interactions"),
    "popular_7d": PopularModel(period=timedelta(days=7)),
    "random": RandomModel(random_state=RANDOM_STATE),
    # "most_raited": PopularModel(popularity="mean_weight"),
    "pop_cat": PopularInCategoryModel(category_feature='genre', n_categories=20),
    # "cosine_knn": ImplicitItemKNNWrapperModel(CosineRecommender()),
    # 'iALS':ImplicitALSWrapperModel(
    #       AlternatingLeastSquares(
    #         factors=10,  # latent embeddings size
    #         regularization=0.1,
    #         iterations=10,
    #         alpha=50,  # confidence multiplier for non-zero entries in interactions
    #         random_state=RANDOM_STATE)),
    'LightFM':LightFMWrapperModel(
            LightFM(no_components=32,
                    loss="warp",
                    random_state=RANDOM_STATE)),

}

# We will calculate several classic (precision@k and recall@k) and "beyond accuracy" metrics
metrics = {
    "prec@10": Precision(k=10),
    "recall@10": Recall(k=10),
    "novelty@10": MeanInvUserFreq(k=10),
    # "serendipity@10": Serendipity(k=10),
    "MAP@10": MAP(k=10),
    "NDCG@10": NDCG(k=10),
    "HitRate@10": HitRate(k=10)
}

K_RECS = 10

In [ ]:
results = []
K_RECOS = 10
RANDOM_STATE = 42
NUM_THREADS =-1
for model_name, model in tqdm(models.items()):
    model_quality = {'model': model_name}

    model.fit(dataset_train_i_u_features)
    recos = model.recommend(
        users=test_cold[Columns.User].unique(),
        dataset=dataset_train_i_u_features, #dataset_feature_train train_dataset
        k=K_RECOS,
        filter_viewed=True,
    )

    metric_values = calc_metrics(metrics, recos, test_cold, train)

    model_quality.update(metric_values)
    results.append(model_quality)

In [ ]:
df_quality = pd.DataFrame(results).T

df_quality.columns = df_quality.iloc[0]

df_quality.drop('model', inplace=True)
# df_quality.style.highlight_max(color='lightgreen', axis=1)
df_quality

In [ ]:
n_splits = 3

splitter = TimeRangeSplitter(
    test_size="7D",
    n_splits=n_splits,
    filter_already_seen=True,
    filter_cold_items=False,
    filter_cold_users=False,
)
cv_results = cross_validate(
    dataset=dataset_train_i_u_features,
    splitter=splitter,
    models=models,
    metrics=metrics,
    k=K_RECS,
    filter_viewed=True,
)

In [ ]:
pivot_results = (
    pd.DataFrame(cv_results["metrics"])
    .drop(columns="i_split")
    .groupby(["model"], sort=False)
    .agg(["mean"])
)
pivot_results

In [ ]:
users_clise

In [ ]:
items_clise

In [ ]:
# pip install sentence-transformers faiss-cpu pandas numpy
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

# # ====== paths ======
# USERS_PATH = "users_close.csv"
# ITEMS_PATH = "items.csv"

# ====== columns (change if your CSV uses different names) ======
U_USER_ID, U_AGE, U_INCOME, U_SEX, U_KIDS = "user_id", "age", "income", "sex", "kids_flg"
I_ITEM_ID, I_TITLE, I_DESC, I_GENRES = "item_id", "title", "description", "keywords"

TOPK = 10
MODEL_NAME = "ai-forever/FRIDA"

# ====== load data ======
users = users_clise.head(1000)
items = items_clise.head(1000)

users[U_USER_ID] = users[U_USER_ID].astype(str)
items[I_ITEM_ID] = items[I_ITEM_ID].astype(str)

# if some text columns missing -> empty
for c in [I_TITLE, I_DESC, I_GENRES]:
    if c not in items.columns:
        items[c] = ""
    items[c] = items[c].fillna("").astype(str)

# ====== FRIDA model ======
model = SentenceTransformer(MODEL_NAME)

# ====== build item "documents" and embed ======
item_docs = (
    "search_document: title: " + items[I_TITLE] +
    "\ngenres: " + items[I_GENRES] +
    "\ndescription: " + items[I_DESC]
).tolist()

item_emb = model.encode(item_docs, normalize_embeddings=True, show_progress_bar=True)
item_emb = np.asarray(item_emb, dtype="float32")

item_ids = items[I_ITEM_ID].values

# ====== FAISS index over items (cosine via inner product on normalized vectors) ======
index = faiss.IndexFlatIP(item_emb.shape[1])
index.add(item_emb)

faiss.write_index(index, "items_frida.index")
np.save("items_frida_ids.npy", item_ids)

# ====== make user queries from demographics and embed ======
def user_query(row):
    return (
        "search_query: Подбери фильмы/сериалы под пользователя: "
        f"пол={row[U_SEX]}, возраст={row[U_AGE]}, доход={row[U_INCOME]}, дети={row[U_KIDS]}. "
        "Учитывай возрастные ограничения и семейный просмотр."
    )

user_queries = users.apply(user_query, axis=1).tolist()
user_emb = model.encode(user_queries, normalize_embeddings=True, show_progress_bar=True)
user_emb = np.asarray(user_emb, dtype="float32")

# ====== search topK items for each user ======
scores, idx = index.search(user_emb, TOPK)          # (n_users, TOPK)
rec_item_ids = item_ids[idx]                        # (n_users, TOPK)

# ====== save recommendations ======
out = pd.DataFrame({
    U_USER_ID: users[U_USER_ID].values,
    "recommended_items": [" ".join(row.tolist()) for row in rec_item_ids],
})
out.to_csv("recommendations.csv", index=False)

print(out.head(10))

In [ ]:
# pip install scikit-learn faiss-cpu pandas numpy
import numpy as np
import pandas as pd
import faiss
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

U_USER_ID, U_AGE, U_INCOME, U_SEX, U_KIDS = "user_id", "age", "income", "sex", "kids_flg"
I_ITEM_ID, I_TITLE, I_DESC, I_GENRES = "item_id", "title", "description", "keywords"

TOPK = 10
SVD_DIM = 256

users = users_clise.head(10000)[train_in_all['user_id'].tolist()[:10000]]
items = items_clise.head(10000)

users[U_USER_ID] = users[U_USER_ID].astype(str)
items[I_ITEM_ID] = items[I_ITEM_ID].astype(str)

# ensure text cols exist
for c in [I_TITLE, I_DESC, I_GENRES]:
    if c not in items.columns:
        items[c] = ""
    items[c] = items[c].fillna("").astype(str)

for c in [U_AGE, U_INCOME, U_SEX, U_KIDS]:
    if c not in users.columns:
        users[c] = ""
    users[c] = users[c].fillna("").astype(str)

# ====== TF-IDF docs for ITEMS ======
item_docs = (
    "title " + items[I_TITLE] + " "
    "keywords " + items[I_GENRES] + " "
    "description " + items[I_DESC]
).tolist()

tfidf = TfidfVectorizer(
    lowercase=True,
    min_df=2,
    max_features=300_000,
    ngram_range=(1, 2),
)
X_items = tfidf.fit_transform(item_docs)

svd = TruncatedSVD(n_components=SVD_DIM, random_state=42)
item_emb = svd.fit_transform(X_items).astype("float32")

faiss.normalize_L2(item_emb)
index = faiss.IndexFlatIP(item_emb.shape[1])
index.add(item_emb)

item_ids = items[I_ITEM_ID].values

# ====== TF-IDF queries for USERS (demographics -> text) ======
user_docs = (
    "sex " + users[U_SEX] + " "
    "age " + users[U_AGE] + " "
    "income " + users[U_INCOME] + " "
    "kids " + users[U_KIDS]
).tolist()

X_users = tfidf.transform(user_docs)
user_emb = svd.transform(X_users).astype("float32")
faiss.normalize_L2(user_emb)

scores, idx = index.search(user_emb, TOPK)
rec_item_ids = item_ids[idx]

out = pd.DataFrame({
    U_USER_ID: users[U_USER_ID].values,
    "recommended_items": [" ".join(row.tolist()) for row in rec_item_ids],
})
# out.to_csv("recommendations.csv", index=False)

print(out.head(10))

In [ ]:
# pip install scikit-learn faiss-cpu pandas numpy
import numpy as np
import pandas as pd
import faiss
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# =====================
# REQUIRED INPUTS (must exist in your notebook/session):
#   users_clise: DataFrame with user features
#   items_clise: DataFrame with item features
#   train_in_all: interactions/gt for train users (must include user_id, item_id)
#   test_cold: interactions/gt for test cold users (must include user_id, item_id)
#
# You said you already create:
#   users_train = users_clise[~users_clise['user_id'].isin(test_cold['user_id'])].head(10000)
#   users_test_id = test_cold['user_id']
# =====================

U_USER_ID, U_AGE, U_INCOME, U_SEX, U_KIDS = "user_id", "age", "income", "sex", "kids_flg"
I_ITEM_ID, I_TITLE, I_DESC, I_GENRES = "item_id", "title", "description", "keywords"

TOPK = 10
SVD_DIM = 256
RANDOM_STATE = 42

# ---------- helpers ----------
def ensure_cols(df, cols):
    df = df.copy()
    for c in cols:
        if c not in df.columns:
            df[c] = ""
        df[c] = df[c].fillna("").astype(str)
    return df

def build_item_docs(items: pd.DataFrame) -> list[str]:
    return (
        "title " + items[I_TITLE] + " "
        "keywords " + items[I_GENRES] + " "
        "description " + items[I_DESC]
    ).tolist()

def build_user_docs(users: pd.DataFrame) -> list[str]:
    return (
        "sex " + users[U_SEX] + " "
        "age " + users[U_AGE] + " "
        "income " + users[U_INCOME] + " "
        "kids " + users[U_KIDS]
    ).tolist()

def topk_recs_for_users(user_emb: np.ndarray, item_ids: np.ndarray, index, k=10):
    scores, idx = index.search(user_emb.astype("float32"), k)
    rec_item_ids = item_ids[idx]
    return rec_item_ids, scores

def build_truth_dict(df_gt: pd.DataFrame) -> dict[str, set[str]]:
    # expects columns user_id, item_id
    df_gt = df_gt[[U_USER_ID, I_ITEM_ID]].copy()
    df_gt[U_USER_ID] = df_gt[U_USER_ID].astype(str)
    df_gt[I_ITEM_ID] = df_gt[I_ITEM_ID].astype(str)
    return df_gt.groupby(U_USER_ID)[I_ITEM_ID].apply(lambda s: set(s.tolist())).to_dict()

def precision_recall_ap_ndcg_hitrate_at_k(rec_list, truth_set, k=10):
    if truth_set is None or len(truth_set) == 0:
        return 0.0, 0.0, 0.0, 0.0, 0.0

    rec_k = rec_list[:k]
    hits = [1 if r in truth_set else 0 for r in rec_k]
    hit_count = sum(hits)

    prec = hit_count / k
    rec = hit_count / len(truth_set)
    hitrate = 1.0 if hit_count > 0 else 0.0

    # AP@k
    ap = 0.0
    denom = min(len(truth_set), k)
    if denom > 0:
        cum = 0
        for i, h in enumerate(hits, start=1):
            if h:
                cum += 1
                ap += cum / i
        ap /= denom

    # NDCG@k (binary relevance)
    dcg = 0.0
    for i, h in enumerate(hits, start=1):
        if h:
            dcg += 1.0 / np.log2(i + 1)
    idcg = sum(1.0 / np.log2(i + 1) for i in range(1, min(len(truth_set), k) + 1))
    ndcg = dcg / idcg if idcg > 0 else 0.0

    return prec, rec, ap, ndcg, hitrate

def novelty_at_k(rec_list, item_inv_user_freq, k=10):
    rec_k = rec_list[:k]
    vals = [item_inv_user_freq.get(it, 0.0) for it in rec_k]
    return float(np.mean(vals)) if len(vals) else 0.0

def build_item_inv_user_freq(train_interactions: pd.DataFrame) -> dict[str, float]:
    # MeanInvUserFreq: mean over recommended items of log(N_users / user_count(item))
    df = train_interactions[[U_USER_ID, I_ITEM_ID]].copy()
    df[U_USER_ID] = df[U_USER_ID].astype(str)
    df[I_ITEM_ID] = df[I_ITEM_ID].astype(str)

    n_users = df[U_USER_ID].nunique()
    item_user_cnt = df.groupby(I_ITEM_ID)[U_USER_ID].nunique()
    # add small epsilon not needed because counts >=1 for seen items
    inv = np.log(n_users / item_user_cnt).to_dict()
    return {str(k): float(v) for k, v in inv.items()}

# =====================
# 1) Split users
# =====================
users_train = users_clise[~users_clise[U_USER_ID].isin(test_cold[U_USER_ID])].copy()
users_test = users_clise[users_clise[U_USER_ID].isin(test_cold[U_USER_ID])].copy()

users_train[U_USER_ID] = users_train[U_USER_ID].astype(str)
users_test[U_USER_ID] = users_test[U_USER_ID].astype(str)

# IMPORTANT: для "холодных" в тесте 0 interactions на вход модели,
# но для метрик у нас должен быть ground truth в test_cold (user_id, item_id).
# Это и есть "что они потом посмотрели".

# =====================
# 2) Prepare items (обычно все items, НЕ head(10000))
# =====================
items = items_clise.copy()
items[I_ITEM_ID] = items[I_ITEM_ID].astype(str)

items = ensure_cols(items, [I_TITLE, I_DESC, I_GENRES])
users_train = ensure_cols(users_train, [U_SEX, U_AGE, U_INCOME, U_KIDS])
users_test = ensure_cols(users_test, [U_SEX, U_AGE, U_INCOME, U_KIDS])

# =====================
# 3) Fit TF-IDF + SVD on ITEMS (train-independent; ok because items are known at inference)
# =====================
item_docs = build_item_docs(items)

tfidf = TfidfVectorizer(
    lowercase=True,
    min_df=2,
    max_features=300_000,
    ngram_range=(1, 2),
)
X_items = tfidf.fit_transform(item_docs)

svd = TruncatedSVD(n_components=SVD_DIM, random_state=RANDOM_STATE)
item_emb = svd.fit_transform(X_items).astype("float32")

faiss.normalize_L2(item_emb)
index = faiss.IndexFlatIP(item_emb.shape[1])
index.add(item_emb)

item_ids = items[I_ITEM_ID].values

# =====================
# 4) Embed TEST users -> recommendations
# =====================
test_user_docs = build_user_docs(users_test)
X_test_users = tfidf.transform(test_user_docs)
test_user_emb = svd.transform(X_test_users).astype("float32")
faiss.normalize_L2(test_user_emb)

rec_item_ids, _ = topk_recs_for_users(test_user_emb, item_ids, index, k=TOPK)

# =====================
# 5) Metrics vs ground-truth test_cold
# =====================
# ground truth dict for test users
truth = build_truth_dict(test_cold)

# novelty needs train interactions (use train_in_all)
item_inv_user_freq = build_item_inv_user_freq(train_in_all)

prec_list, rec_list, nov_list, map_list, ndcg_list, hit_list = [], [], [], [], [], []

test_user_ids = users_test[U_USER_ID].values
for i, uid in enumerate(test_user_ids):
    recs = rec_item_ids[i].tolist()
    gt = truth.get(uid, set())

    prec, rec, ap, ndcg, hit = precision_recall_ap_ndcg_hitrate_at_k(recs, gt, k=TOPK)
    nov = novelty_at_k(recs, item_inv_user_freq, k=TOPK)

    prec_list.append(prec)
    rec_list.append(rec)
    map_list.append(ap)
    ndcg_list.append(ndcg)
    hit_list.append(hit)
    nov_list.append(nov)

metrics = {
    "prec@10": float(np.mean(prec_list)) if prec_list else 0.0,
    "recall@10": float(np.mean(rec_list)) if rec_list else 0.0,
    "novelty@10": float(np.mean(nov_list)) if nov_list else 0.0,
    "MAP@10": float(np.mean(map_list)) if map_list else 0.0,
    "NDCG@10": float(np.mean(ndcg_list)) if ndcg_list else 0.0,
    "HitRate@10": float(np.mean(hit_list)) if hit_list else 0.0,
}
print(metrics)

# =====================
# 6) Optional: output predictions for test user ids
# =====================
out = pd.DataFrame({
    U_USER_ID: test_user_ids,
    "recommended_items": [" ".join(row.tolist()) for row in rec_item_ids],
})
print(out.head(10))